## Preparación del dataset de imagen completa para RF-DETR (Colab)

Descomprimimos el zip original y creamos una copia independiente para RF-DETR,
sin tocar el original.

### Paso 1: Montar Drive y descomprimir

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile

with zipfile.ZipFile("/content/drive/MyDrive/TFM DATA SCIENCE/dataset_kanji_prep.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/dataset_original")

print("Descomprimido en /content/dataset_original")

Descomprimido en /content/dataset_original


### Paso 3: Verificar estructura del dataset original

In [3]:
import os

ruta_interna = "/content/dataset_original/dataset_kangi_prep"
for item in os.listdir(ruta_interna):
    print(item)

dataset_original
dataset_denoised


In [4]:
import shutil

# Original
shutil.copytree(
    "/content/dataset_original/dataset_kangi_prep/dataset_original",
    "/content/dataset_kanji_imagen_completa_RF_DTR/originales"
)
print("Copia 'originales' creada")

# Denoised
shutil.copytree(
    "/content/dataset_original/dataset_kangi_prep/dataset_denoised",
    "/content/dataset_kanji_imagen_completa_RF_DTR/denoised"
)
print("Copia 'denoised' creada")

Copia 'originales' creada
Copia 'denoised' creada


In [5]:
for version in ["originales", "denoised"]:
    base = f"/content/dataset_kanji_imagen_completa_RF_DTR/{version}"
    print(f"=== {version} ===")
    print("Carpetas:", sorted(os.listdir(base)))
    print("Dentro de images:", sorted(os.listdir(f"{base}/images")))
    print()

=== originales ===
Carpetas: ['images', 'labels']
Dentro de images: ['test', 'train', 'val']

=== denoised ===
Carpetas: ['images', 'labels']
Dentro de images: ['test', 'train', 'val']



### Paso 6: Renombrar val → valid (solo en la copia RF-DETR)

In [6]:
base = "/content/dataset_kanji_imagen_completa_RF_DTR"

for version in ["originales", "denoised"]:
    for tipo in ["images", "labels"]:
        ruta_val = f"{base}/{version}/{tipo}/val"
        ruta_valid = f"{base}/{version}/{tipo}/valid"
        os.rename(ruta_val, ruta_valid)
        print(f"Renombrado: {ruta_val} -> {ruta_valid}")

Renombrado: /content/dataset_kanji_imagen_completa_RF_DTR/originales/images/val -> /content/dataset_kanji_imagen_completa_RF_DTR/originales/images/valid
Renombrado: /content/dataset_kanji_imagen_completa_RF_DTR/originales/labels/val -> /content/dataset_kanji_imagen_completa_RF_DTR/originales/labels/valid
Renombrado: /content/dataset_kanji_imagen_completa_RF_DTR/denoised/images/val -> /content/dataset_kanji_imagen_completa_RF_DTR/denoised/images/valid
Renombrado: /content/dataset_kanji_imagen_completa_RF_DTR/denoised/labels/val -> /content/dataset_kanji_imagen_completa_RF_DTR/denoised/labels/valid


### Paso 7: Reorganizar estructura (images/train → train/images)

In [7]:
import shutil

base = "/content/dataset_kanji_imagen_completa_RF_DTR"

for version in ["originales", "denoised"]:
    base_v = f"{base}/{version}"

    for split in ["train", "valid", "test"]:
        os.makedirs(f"{base_v}/{split}/images", exist_ok=True)
        os.makedirs(f"{base_v}/{split}/labels", exist_ok=True)

        for archivo in os.listdir(f"{base_v}/images/{split}"):
            shutil.move(f"{base_v}/images/{split}/{archivo}", f"{base_v}/{split}/images/{archivo}")
        for archivo in os.listdir(f"{base_v}/labels/{split}"):
            shutil.move(f"{base_v}/labels/{split}/{archivo}", f"{base_v}/{split}/labels/{archivo}")

    shutil.rmtree(f"{base_v}/images")
    shutil.rmtree(f"{base_v}/labels")

    print(f"{version}: reorganización completada")

originales: reorganización completada
denoised: reorganización completada


### Paso 8: Crear data.yaml para ambas versiones

In [8]:
contenido_yaml = """path: {ruta}
train: train/images
valid: valid/images
test: test/images

names:
  0: kanji
"""

base = "/content/dataset_kanji_imagen_completa_RF_DTR"

for version in ["originales", "denoised"]:
    ruta = f"{base}/{version}"
    with open(f"{ruta}/data.yaml", "w") as f:
        f.write(contenido_yaml.format(ruta=ruta))
    print(f"Creado: {ruta}/data.yaml")

Creado: /content/dataset_kanji_imagen_completa_RF_DTR/originales/data.yaml
Creado: /content/dataset_kanji_imagen_completa_RF_DTR/denoised/data.yaml


In [9]:
for version in ["originales", "denoised"]:
    base = f"/content/dataset_kanji_imagen_completa_RF_DTR/{version}"
    print(f"=== {version} ===")
    print("Carpetas:", sorted(os.listdir(base)))
    print("Dentro de train:", sorted(os.listdir(f"{base}/train")))
    for split in ["train", "valid", "test"]:
        n_img = len(os.listdir(f"{base}/{split}/images"))
        n_lbl = len(os.listdir(f"{base}/{split}/labels"))
        print(f"  {split}: {n_img} imágenes, {n_lbl} labels")
    print()

=== originales ===
Carpetas: ['data.yaml', 'test', 'train', 'valid']
Dentro de train: ['images', 'labels']
  train: 55 imágenes, 55 labels
  valid: 10 imágenes, 10 labels
  test: 10 imágenes, 10 labels

=== denoised ===
Carpetas: ['data.yaml', 'test', 'train', 'valid']
Dentro de train: ['images', 'labels']
  train: 55 imágenes, 55 labels
  valid: 10 imágenes, 10 labels
  test: 10 imágenes, 10 labels



In [10]:
!pip install rfdetr --quiet
!pip install "rfdetr[train,loggers]" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 150.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Paso 11: Entrenamiento - Imagen Completa Original

In [11]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

model.train(
    dataset_dir="/content/dataset_kanji_imagen_completa_RF_DTR/originales",
    epochs=50,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    early_stopping_patience=10,
    seed=42,
    output_dir="/content/resultados_RF_DTR/imagen_completa_original_50ep_RF_DTR"
)

Val — Overall Metrics                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.3006 │ 0.5878 │ 0.2739 │ 0.3383 │ 0.7003 │ 0.8744 │ 0.5840 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘
                  Val — Per-class Metrics                  
┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ kanji │   0.3006 │ 0.3383 │ 0.7003 │    0.8744 │ 0.5840 │
└───────┴──────────┴────────┴────────┴───────────┴────────┘

[2026-06-23 20:02:32] [INFO] rf-detr - Best EMA mAP improved to 0.3059 (epoch 49)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


[2026-06-23 20:02:38] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.3032, ema=0.3059)


In [ ]:
import shutil

shutil.make_archive(
    "/content/dataset_kanji_imagen_completa_RF_DTR",
    "zip",
    "/content/",
    "dataset_kanji_imagen_completa_RF_DTR"
)

from google.colab import files
files.download("/content/dataset_kanji_imagen_completa_RF_DTR.zip")

### Entrenamiento - RF-DETR Imagen Completa Denoised

In [15]:
from rfdetr import RFDETRSmall

model_denoised = RFDETRSmall()

model_denoised.train(
    dataset_dir="/content/dataset_kanji_imagen_completa_RF_DTR/denoised",
    epochs=50,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    early_stopping_patience=10,
    seed=42,
    output_dir="/content/resultados_RF_DTR/imagen_completa_denoised_50ep_RF_DTR"
)

[2026-06-23 20:23:34] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-06-23 20:23:34] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 20:23:34] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-23 20:23:35] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-06-23 20:23:36] [WARNING] rf-detr - Pretrained weights at '/root/.roboflow/models/rf-detr-small.pth' loaded only partially — this typically produces lower accuracy. 1 model parameter(s) not in checkpoint (left at random init): [_kp_active_mask]. Check that the model configuration (encoder, hidden_dim, out_feature_indexes, projector_scale, ...) matches the architecture the checkpoint was trained with.
[2026-06-23 20:23:36] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-23 20:23:36] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-23 20:23:37] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


...

INFO:lightning_fabric.utilities.seed:Seed set to 42


Output()

[2026-06-23 20:24:22] [INFO] rf-detr - Best regular checkpoint saved to /content/resultados_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_regular.pth (epoch 0, monitor=val/mAP_50_95, value=0)
[2026-06-23 20:25:07] [INFO] rf-detr - Best regular checkpoint saved to /content/resultados_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_regular.pth (epoch 1, monitor=val/mAP_50_95, value=0.0371046)
[2026-06-23 20:25:07] [INFO] rf-detr - Best EMA mAP improved to 0.0367 (epoch 1)
[2026-06-23 20:26:37] [INFO] rf-detr - Best regular checkpoint saved to /content/resultados_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_regular.pth (epoch 3, monitor=val/mAP_50_95, value=0.0456736)
[2026-06-23 20:26:37] [INFO] rf-detr - Best EMA mAP improved to 0.0410 (epoch 3)
[2026-06-23 20:27:21] [INFO] rf-detr - Best regular checkpoint saved to /content/resultados_RF_DTR/imagen_completa_denoised_50ep_RF_DTR/checkpoint_best_regular.pth (epoch 4, monitor=val/mAP_50_95, value=0

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


[2026-06-23 21:00:59] [INFO] rf-detr - Best total checkpoint saved from regular (regular=0.1822, ema=0.1800)
